In [1]:
import sqlite3
import pandas as pd
import os
from mappings import resolve_company_sec, fetch_sec_tickers, get_biotech_universe, clean_name
from scraper import fetch_trials
import requests

save_dir = 'C:\\Users\\chris\Documents\\personal_projects'
os.chdir(save_dir)

conn = sqlite3.connect("biotech.db")

df = pd.read_sql_query("""
    WITH latest_trial AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY nct_id ORDER BY snapshot_date DESC) AS rn
        FROM trials
    ),
    latest_pub AS (
        SELECT nct_id, title AS latest_publication, pub_date,
               ROW_NUMBER() OVER (PARTITION BY nct_id ORDER BY pub_date DESC) AS rn
        FROM publications
    )
    SELECT t.company, t.nct_id, t.phase, t.status, t.conditions,
           t.primary_completion_date, t.enrollment,
           p.latest_publication, p.pub_date
    FROM latest_trial t
    LEFT JOIN latest_pub p
        ON t.nct_id = p.nct_id AND p.rn = 1
    WHERE t.rn = 1
      AND t.phase IN ('PHASE3', 'PHASE4')
      AND t.status = 'ACTIVE_NOT_RECRUITING'
      AND t.primary_completion_date BETWEEN date('now') AND date('now', '+6 months')
    ORDER BY t.primary_completion_date
""", conn)

print(len(df))
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
df.sort_values('primary_completion_date')[['company', 'phase', 'primary_completion_date', 'conditions', 'enrollment', 'nct_id', 'latest_publication']].head(40)
#pd.reset_option('display.max_colwidth')
#pd.reset_option('display.max_rows')

139


,company,phase,primary_completion_date,conditions,enrollment,nct_id,latest_publication
0,sanofi,PHASE4,2026-06-26,Type 2 Diabetes,105,NCT06716424,None
1,sanofi,PHASE3,2026-06-29,Dermatitis Atopic,1541,NCT06407934,The Role of OX40 Pathway Inhibition as a New Therapeutic Strategy for Atopic Dermatitis.
2,novo nordisk,PHASE3,2026-06-29,Type 2 Diabetes,264,NCT07271251,None
15,akebia therapeutics,PHASE3,2026-06-30,Anemia of Chronic Kidney Disease,2200,NCT06520826,None
14,cg oncology,PHASE3,2026-06-30,"Non Muscle Invasive Bladder Cancer,Urologic Cancer,Bladder Cancer,Urothelial Carcinoma",367,NCT06111235,None
13,sanofi,PHASE3,2026-06-30,Symptomatic Non-Obstructive Hypertrophic Cardiomyopathy,500,NCT06081894,Efficacy and Safety of Aficamten in Children and Adolescents With Obstructive Hypertrophic Cardiomyopathy: Study Design and Rationale of CEDAR-HCM.
12,biontech,PHASE3,2026-06-30,Metastatic Breast Cancer,541,NCT06018337,Antibodies to watch in 2026.
11,merck,PHASE3,2026-06-30,HIV-1-infection,600,NCT05924438,Brief Report: Resolution of Neuropsychiatric Adverse Events After Switching to a Doravirine-Based Regimen in the Open-Label Extensions of the DRIVE-AHEAD and DRIVE-FORWARD Trials.
10,zenas biopharma,PHASE3,2026-06-30,Warm Autoimmune Hemolytic Anemia,134,NCT05786573,"Beneath the surface in autoimmune hemolytic anemia: pathogenetic networks, therapeutic advancements and open questions."
8,celcuity,PHASE3,2026-06-30,Breast Cancer,701,NCT05501886,Beyond CDK4/6 Inhibition: Current Strategies in Hormone Receptor-Positive Metastatic Breast Cancer.
